In [2]:
%load_ext autoreload

%autoreload 2

In [3]:
from typing import Dict, List, Optional, Tuple, Union, Any, Callable, Mapping
import pandas as pd
import pickle
import numpy as np
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, matthews_corrcoef, balanced_accuracy_score, roc_auc_score, cohen_kappa_score, auc, precision_recall_curve)
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier as xgb

In [4]:
def fin(df, radius, fpSize):
    fingerprints = []
    onbits_list = []
    fp_generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fpSize)
    for i, mol in enumerate(df["ROMol"]):
        try:
            fp = fp_generator.GetFingerprint(mol)
            # 1になっているビットの位置を取得
            onbits = list(fp.GetOnBits())
            onbits_list.append(onbits)
            
            # NumPy配列も必要なら
            fp_np = fp_generator.GetFingerprintAsNumPy(mol)
            fingerprints.append(fp_np)

        except Exception as e:
            print(f"Error processing molecule {i}: {e}")
            continue
    return np.array(fingerprints), onbits_list

def add_vectors(fp_list: List[List[int]], model: Doc2Vec) -> List[np.ndarray]:
    """Combine document vectors based on fingerprints
    
    Args:
        fp_list: List of fingerprint lists, where each fingerprint is represented as a list of indices
        model: Trained Doc2Vec model containing document vectors
        
    Returns:
        List of compound vectors as numpy arrays
    """
    compound_vec = []
    for i in fp_list:
        fingerprint_vec = 0
        for j in i:
            fingerprint_vec += model.dv.vectors[j]
        compound_vec.append(fingerprint_vec)
    return compound_vec

def calculate_metrics(y_true, y_pred, y_proba):

    metrics = {}
    metrics['f1'] = f1_score(y_true, y_pred)
    metrics['mcc'] = matthews_corrcoef(y_true, y_pred)
    metrics['balanced_accuracy'] = balanced_accuracy_score(y_true, y_pred)
    metrics['roc_auc'] = roc_auc_score(y_true, y_proba)
    metrics['kappa'] = cohen_kappa_score(y_true, y_pred)
    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    metrics['pr_auc'] = auc(recall, precision)
    return metrics

def evaluate_category(X_vec: np.ndarray, 
                      y: np.ndarray, 
                      classifier
                      ) -> Dict[str, Union[List[float], float]]:
    
    # 全ての評価指標のスコアを格納する辞書
    all_train_scores = {'f1': [], 'mcc': [], 'balanced_accuracy': [], 'roc_auc': [], 'kappa': [], 'pr_auc': []}
    all_test_scores = {'f1': [], 'mcc': [], 'balanced_accuracy': [], 'roc_auc': [], 'kappa': [], 'pr_auc': []}


    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    for train_idx, test_idx in skf.split(range(len(y)), y):
        X_train_vec, X_test_vec = X_vec[train_idx], X_vec[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        classifier.fit(X_train_vec, y_train)
        y_train_pred = classifier.predict(X_train_vec)
        y_test_pred = classifier.predict(X_test_vec)
        y_train_proba = classifier.predict_proba(X_train_vec)[:, 1]
        y_test_proba = classifier.predict_proba(X_test_vec)[:, 1]

        train_metrics = calculate_metrics(y_train, y_train_pred, y_train_proba)
        test_metrics = calculate_metrics(y_test, y_test_pred, y_test_proba)
        for metric_name in all_train_scores.keys():
            all_train_scores[metric_name].append(train_metrics[metric_name])
            all_test_scores[metric_name].append(test_metrics[metric_name])
    
    # 結果を整理
    results = {}
    for metric_name in all_train_scores.keys():
        results[metric_name] = {
            'train_scores': all_train_scores[metric_name],
            'test_scores': all_test_scores[metric_name],
            'mean_train': np.mean(all_train_scores[metric_name]),
            'mean_test': np.mean(all_test_scores[metric_name])
        }
    
    return results

def main(input_path: str,
         model_path: str,
         radius: int,
         fpSize: int,
         classifier) -> Dict[str, Dict[str, float]]:
    """
    Main function to train and evaluate compound classification models using provided features and Doc2Vec.
    
    Args:
        input_path: Path to the pickle file containing compound data
        feature_list: List of molecular features (like fingerprints) to use in the model
        doc2vec_param: Parameters for the Doc2Vec model
        lightgbm_model: Pre-configured LightGBM classifier
        purpose_description: Column name in the DataFrame containing text descriptions
        
    Returns:
        Dictionary mapping category names to evaluation results
    """
        
    # Define categories to evaluate
    categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
    with open(input_path, "rb") as f:
        df = pickle.load(f)
    model = Doc2Vec.load(model_path)
    bit_list = fin(df, radius, fpSize)[1]
    compound_vec = add_vectors(bit_list, model)
    X_vec = np.array(compound_vec)

    # Evaluate each category
    results = {}
    for category in categories:
        y = np.array([1 if i == category else 0 for i in df[category]])
        results[category] = evaluate_category(X_vec, y, classifier)

    return results

!!! light gbm !!!

In [13]:
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

gbm_params: Dict[str, Any] = {
    "boosting_type": "dart",
    "num_leaves": 48,
    "max_depth": 5,
    "learning_rate": 0.04166324251391809,
    "n_estimators": 736,
    "class_weight": "balanced",
    "min_split_gain": 0.009346925180781129,
    "min_child_weight": 0.0007929549087822909,
    "min_child_samples": 37,
    "reg_alpha": 1.757104268180148,
    "reg_lambda": 1.463369722508726,
    "feature_fraction": 0.50163362868711,
    "feature_fraction_bynode": 0.8321043377994284,
    "subsample": 0.6974385909748512,
    "colsample_bytree": 0.6568268046410831,
    "subsample_freq": 5,
    "drop_rate": 0.24668126335938073,
    "max_drop": 28,
    "skip_drop": 0.5591506516119614,
    "uniform_drop": True,
    "xgboost_dart_mode": True,
    "objective": "binary",
    "random_state": 0,
    "verbose": -1,
    "force_col_wise": True
}

# Create classifier
lightgbm_model = lgb.LGBMClassifier(**gbm_params)

# Load dataset for later use
# Example usage - replace with your actual file paths
input_path = "data/train_df2.pkl"
model_path = "train_df_doc2vec.model"
lightgbm_results = main(input_path, model_path, 3, 4096, lightgbm_model)

In [18]:
li = []
for category, result in lightgbm_results.items():
    print(f"## {category} ##")
    print(lightgbm_results[category]['mcc']["mean_test"])
    li.append(lightgbm_results[category]['mcc']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.6825388153946739
## anti_inflammatory_agent ##
0.6956165157526341
## allergen ##
0.6619015583523324
## dye ##
0.9267184241965426
## toxin ##
0.6174931945910888
## flavouring_agent ##
0.7180149889064215
## agrochemical ##
0.7893064227493397
## volatile_oil ##
0.7944544906364606
## antibacterial_agent ##
0.6347927623112386
## insecticide ##
0.7548672243802339

0.7275704397270966


In [ ]:
with open("result_classifier_change/LightGBM.pkl", "wb") as f:
    pickle.dump(lightgbm_results, f)

In [7]:
# {'f1': [], 'mcc': [], 'balanced_accuracy': [], 'roc_auc': [], 'kappa': [], 'pr_auc': []}

with open("result_classifier_change/LightGBM.pkl", "rb") as f:
    a = pickle.load(f)
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in a.items():
    print(f"## {category} ##")
    print(a[category]['mcc']["mean_test"])
    li.append(a[category]['mcc']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.6825388153946739
## anti_inflammatory_agent ##
0.6956165157526341
## allergen ##
0.6619015583523324
## dye ##
0.9267184241965426
## toxin ##
0.6174931945910888
## flavouring_agent ##
0.7180149889064215
## agrochemical ##
0.7893064227493397
## volatile_oil ##
0.7944544906364606
## antibacterial_agent ##
0.6347927623112386
## insecticide ##
0.7548672243802339

0.7275704397270966


!!! adaboost_decision !!!

In [6]:
dt_params = {
    "max_depth": 3,
    "min_samples_split": 13,
    "min_samples_leaf": 7,
    "max_features": None,
    "class_weight": None
}
dt = DecisionTreeClassifier(**dt_params)

ada_params = {
    "n_estimators": 780,
    "learning_rate": 0.4026917217094199,
    "random_state": 0
}
ada = AdaBoostClassifier(estimator=dt, **ada_params)

# Load dataset for later use
# Example usage - replace with your actual file paths
input_path = "data/train_df2.pkl"
model_path = "train_df_doc2vec.model"
ada_results = main(input_path, model_path, 3, 4096, ada)

In [8]:
li = []
for category, result in ada_results.items():
    print(f"## {category} ##")
    print(ada_results[category]['mcc']["mean_test"])
    li.append(ada_results[category]['mcc']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.6584685596341957
## anti_inflammatory_agent ##
0.6677253842177924
## allergen ##
0.6151453498531249
## dye ##
0.9255064382468134
## toxin ##
0.5757717242861805
## flavouring_agent ##
0.6677806037224824
## agrochemical ##
0.7757043647713144
## volatile_oil ##
0.7750210168045972
## antibacterial_agent ##
0.6035061056168566
## insecticide ##
0.7646786619144249

0.7029308209067782


In [9]:
with open("result_classifier_change/AdaBoost.pkl", "wb") as f:
    pickle.dump(ada_results, f)

!!! Extra Tree !!!

In [10]:
et_params = {
        "n_estimators": 223, 
        "criterion": "entropy", 
        "max_depth": 48, 
        "min_samples_split": 28, 
        "min_samples_leaf": 1, 
        "min_weight_fraction_leaf": 9.461512548005503e-05, 
        "max_features": None, 
        "bootstrap": False, 
        "class_weight": "balanced_subsample", 
        "min_impurity_decrease": 0.00109436748269209,
        "ccp_alpha": 0.00023066653634845876,
        "random_state": 0,
        "n_jobs" : 1,
        "verbose" : 0
}

et = ExtraTreesClassifier(**et_params)

# Load dataset for later use
# Example usage - replace with your actual file paths
input_path = "data/train_df2.pkl"
model_path = "train_df_doc2vec.model"
et_results = main(input_path, model_path, 3, 4096, et)

In [11]:
li = []
for category, result in et_results.items():
    print(f"## {category} ##")
    print(et_results[category]['mcc']["mean_test"])
    li.append(et_results[category]['mcc']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.6294238219296957
## anti_inflammatory_agent ##
0.674013523279757
## allergen ##
0.6070231515867991
## dye ##
0.9067000700234367
## toxin ##
0.6055267687084912
## flavouring_agent ##
0.6586751026499955
## agrochemical ##
0.7585163007688575
## volatile_oil ##
0.7690604444124972
## antibacterial_agent ##
0.5921422755258132
## insecticide ##
0.7161863474302643

0.6917267806315607


In [12]:
with open("result_classifier_change/ExtraTrees.pkl", "wb") as f:
    pickle.dump(et_results, f)

!!! Xg boost !!!

In [27]:
xgb_params: Dict[str, Any] = {
    "n_estimators": 810, 
    "max_depth": 9,
    "learning_rate": 0.0750274690444854, 
    "booster": "gbtree",
    "gamma": 0.06006649475004644, 
    "min_child_weight": 6.525033114236552,
    "max_delta_step": 4.3290483461345115, 
    "subsample": 0.6884228113550477,
    "colsample_bytree": 0.7444224692655531, 
    "reg_alpha": 1.7329549753275677,
    "reg_lambda": 4.33369477797301, 
    "scale_pos_weight": 9.91504160087765, 
    "random_state": 0,
    "n_jobs": 1,
    "eval_metric": 'logloss',
    "verbosity": 0
}

xg_boost = xgb(**xgb_params)

# Load dataset for later use
# Example usage - replace with your actual file paths
input_path = "data/train_df2.pkl"
model_path = "train_df_doc2vec.model"
xg_boost_results = main(input_path, model_path, 3, 4096, xg_boost)

In [28]:
li = []
for category, result in xg_boost_results.items():
    print(f"## {category} ##")
    print(xg_boost_results[category]['mcc']["mean_test"])
    li.append(xg_boost_results[category]['mcc']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.6938722608533012
## anti_inflammatory_agent ##
0.6965582344468695
## allergen ##
0.656776368577936
## dye ##
0.930030852430671
## toxin ##
0.611829149777909
## flavouring_agent ##
0.6850846533624717
## agrochemical ##
0.8108127935518592
## volatile_oil ##
0.7949171723444006
## antibacterial_agent ##
0.6418403891226017
## insecticide ##
0.7616560314971774

0.7283377905965198


In [29]:
with open("result_classifier_change/XGBoost.pkl", "wb") as f:
    pickle.dump(xg_boost_results, f)

!!! Randam forest !!!

In [31]:
rf_params: Dict[str, Any] = {
       "n_estimators": 815, 
       "criterion": "log_loss", 
       "max_depth": 26, 
       "min_samples_split": 13, 
       "min_samples_leaf": 9, 
       "min_weight_fraction_leaf": 0.0023093752415075915, 
       "max_features": "log2", 
       "bootstrap": False, 
       "class_weight": "balanced_subsample", 
       "random_state": 50
}

rf = RandomForestClassifier(**rf_params)

# Load dataset for later use
# Example usage - replace with your actual file paths
input_path = "data/train_df2.pkl"
model_path = "train_df_doc2vec.model"
rf_results = main(input_path, model_path, 3, 4096, rf)

In [32]:
li = []
for category, result in rf_results.items():
    print(f"## {category} ##")
    print(rf_results[category]['mcc']["mean_test"])
    li.append(rf_results[category]['mcc']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.6207926991893208
## anti_inflammatory_agent ##
0.6719743248584942
## allergen ##
0.6068246264352226
## dye ##
0.9084865217220874
## toxin ##
0.5520299102398166
## flavouring_agent ##
0.639710325310179
## agrochemical ##
0.7512211994096893
## volatile_oil ##
0.7958807774144464
## antibacterial_agent ##
0.5920863028076575
## insecticide ##
0.7143216032645242

0.6853328290651438


In [33]:
with open("result_classifier_change/RF.pkl", "wb") as f:
    pickle.dump(rf_results, f)

!!! Logistic Regression !!!

In [5]:
lr_params: Dict[str, Any] = {
       "C": 0.001899363614004594, 
       "penalty": "l2", 
       "max_iter": 5000, 
       "class_weight": None, 
       "tol": 0.0002926347374178257, 
       "solver": "newton-cg", 
       "random_state": 0
}

lr = LogisticRegression(**lr_params)

# Load dataset for later use
# Example usage - replace with your actual file paths
input_path = "data/train_df2.pkl"
model_path = "train_df_doc2vec.model"
lr_results = main(input_path, model_path, 3, 4096, lr)

/opt/homebrew/anaconda3/envs/fpdoc2vec/lib/python3.11/site-packages/sklearn/utils/optimize.py:100: LineSearchWarning: The line search algorithm did not converge
  ret = line_search_wolfe2(
/opt/homebrew/anaconda3/envs/fpdoc2vec/lib/python3.11/site-packages/sklearn/utils/optimize.py:312: UserWarning: Line Search failed
  warnings.warn("Line Search failed")


In [8]:
lr_params: Dict[str, Any] = {
       "C": 5.378351934170314, 
       "penalty": "l1", 
       "max_iter": 4300, 
       "class_weight": None, 
       "tol": 0.003446628082848086, 
       "solver": "liblinear", 
       "random_state": 0
}

lr = LogisticRegression(**lr_params)

# Load dataset for later use
# Example usage - replace with your actual file paths
input_path = "data/train_df2.pkl"
model_path = "train_df_doc2vec.model"
lr_results = main(input_path, model_path, 3, 4096, lr)

In [6]:
li = []
for category, result in lr_results.items():
    print(f"## {category} ##")
    print(lr_results[category]['f1']["mean_test"])
    li.append(lr_results[category]['f1']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.7377789786360516
## anti_inflammatory_agent ##
0.7766669735706032
## allergen ##
0.793562544859802
## dye ##
0.9257761775209545
## toxin ##
0.5320830413367726
## flavouring_agent ##
0.7771668918436092
## agrochemical ##
0.823108001991723
## volatile_oil ##
0.8075342263288057
## antibacterial_agent ##
0.7490438249683988
## insecticide ##
0.8075531794194809

0.7730273840476201


In [15]:
li = []
for category, result in lr_results.items():
    print(f"## {category} ##")
    print(lr_results[category]['f1']["mean_train"])
    li.append(lr_results[category]['f1']["mean_train"])
print("")
print(np.mean(li))

## antioxidant ##
0.8419352597633394
## anti_inflammatory_agent ##
0.8362839247800554
## allergen ##
0.8982235498319632
## dye ##
0.9822910598590822
## toxin ##
0.7244474256296958
## flavouring_agent ##
0.9528381954036156
## agrochemical ##
0.9085053433652662
## volatile_oil ##
0.9609729016846386
## antibacterial_agent ##
0.8187981946534869
## insecticide ##
0.8977416758426336

0.8822037530813777


In [20]:
with open("result_classifier_change/LR.pkl", "wb") as f:
    pickle.dump(lr_results, f)

In [20]:
with open("result_classifier_change/RF.pkl", "rb") as f:
    a = pickle.load(f)

In [21]:
li = []
for category, result in a.items():
    print(f"## {category} ##")
    print(a[category]['f1']["mean_test"])
    li.append(a[category]['f1']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.6606952926427015
## anti_inflammatory_agent ##
0.7158528077007272
## allergen ##
0.6353304860547293
## dye ##
0.9210074048423451
## toxin ##
0.5482736808341555
## flavouring_agent ##
0.6551410501019742
## agrochemical ##
0.7817393378321759
## volatile_oil ##
0.8045219638242894
## antibacterial_agent ##
0.6466615902712739
## insecticide ##
0.727613816198932

0.7096837430303304
